# Scales

Uma scale (escala de medição) define como os dados são representados e quais operações matemáticas fazem sentido com eles. Temos quatro tipos principais de escalas. 

1) Nominal Scale <br>
É a escala mais simples. Os valores apenas classificam ou identificam categorias, sem ordem entre elas. Dessa forma, não é possível dizer que uma categoria é 'maior' ou 'menor' que outra. 

Exemplo: Cor dos olhos,tipo de fruta, país, gênero. <br>
Operações possíveis: Apenas comparação de igualdade (igual ou diferente)

2) Ordinal Scale <br>
Os valores possuem ordem ou rankin, mas a distância entre eles não é conhecida ou não é consistente. 

Exemplo: Posições em uma corrida (1°, 2°, 3°), nível de satisfação (ruim, médio, bom), classificação de hotéis (3 estrelas, 4 estrelas). <br>
Operações possíveis: comparar ordem (maior/menor), mas não faz sentido calcular diferenças ou médias confiáveis.

3) Interval Scale <br>
Os valores possuem ordem e intervalos iguais entre eles, ou seja, a diferença entre valores é significativa. Porém, não existe um zero absoluto (zero não significa ausência da quantidade).

Exemplos: temperatura em °C ou °F, anos do calendário. <br>
Operações possíveis: adição e subtração, mas não faz sentido falar em proporções (por exemplo, 20°C não é “o dobro” de 10°C).

4) Ratio Scale <br>
É a escala mais completa. Possui ordem, intervalos iguais e zero absoluto, que representa ausência da quantidade medida.

Exemplos: peso, altura, idade, distância, tempo. <br>
Operações possíveis: todas as operações matemáticas, incluindo proporções (ex.: 20 kg é o dobro de 10 kg).

### O que isso tem haver com tipos de dados? 

Tem relação direta, mas não são exatamente a mesma coisa. <br>
Os tipos de dados descrevem a natureza dos valores, enquanto as scales (escalas de medição) descrevem o que podemos fazer matematicamente com esses valores.



In [1]:
import pandas as pd

### Mas por que estamos falando sobre isso?

As escalas são muito importantes em estatística e machine learning, e, por conta disso, Pandas tem várias funções para lidar com a conversão entre as escalas de medida. 

- Nominal Datas (Dados Categóricos): O Pandas tem um tipo para dados categóricos

In [48]:
# Criando um DataFrame
df = pd.DataFrame({
    'nome': ['Ana', 'Bruno', 'Carlos', 'Julia'],
    'classificacao': ['Excelente', 'Médio', 'Ruim', 'Bom']
})

df.dtypes

nome             str
classificacao    str
dtype: object

In [39]:
# Convertendo para categórico
df['classificacao'] = df['classificacao'].astype('category')

df.dtypes

nome                  str
classificacao    category
dtype: object

In [43]:
# Nossos dados podem ser ordenados, ou seja, um bom vem depois de um médio, e um exclente vem depois de um bom. 
minha_categoria = pd.CategoricalDtype(categories = ['Ruim', 'Médio', 'Bom', 'Excelente'],
                                      ordered = True)

# Agora, na função 'astype()', ao invés de passar uma string, podemos passar essa CategoricalDtype
classificacao = df['classificacao'].astype(minha_categoria)
classificacao.head()

0    Excelente
1        Médio
2         Ruim
3          Bom
Name: classificacao, dtype: category
Categories (4, str): ['Ruim' < 'Médio' < 'Bom' < 'Excelente']

Perceba que o Pandas está ciente da ordem dessas categorias. 

In [45]:
# O que podemos fazer com essa ordem? 
# Como temos uma ordenação, isso pode ajudar com comparações e mascaramento booleano. Por exemplo, se temos uma lista das classificações e comparamos com um 'Ruim', podemos verificar quais comparações lexicográficas retornarão resultados que esperamos. 

# Vamos levar a nossa lista das categorias para o dataframe primeiro
df['classificacao'] = df['classificacao'].astype(minha_categoria)

df[df['classificacao'] > 'Ruim']

,nome,classificacao
0,Ana,Excelente
1,Bruno,Médio
3,Julia,Bom


Perceba que agora tem uma ordem

In [51]:
# OBS: Podemos fazer isso para uma coluna em específico sem estar com a forma padronizada igual é feito com pd.CategoricalDtype, mas sim apenas pd.Categorical:
df['classificacao'] = pd.Categorical(
    df['classificacao'],
    categories=['Ruim', 'Médio', 'Bom', 'Excelente'],
    ordered=True
)

df[df['classificacao'] > 'Ruim']

,nome,classificacao
0,Ana,Excelente
1,Bruno,Médio
3,Julia,Bom


ATENÇÃO: A forma com CategoricalDtype é mais utilizada por ser mais padrão, pois se tivermos colunas com valores que possuem a mesma ordem, é mais fácil de reutilizar. 

Basicamente com CategoricalDtype, estamos criando um tipo categórico que carrega a ordem, o qual podemos modificar qualquer dataframe com astype(), já o Categorical, estamos alterando o tipo na própria coluna.

Com a ordem, também podemos utilizar operadores matemáticos, como mínimo, máximo, etc. 

In [52]:
# Às vezes, é util representar cada valor categórico como uma coluna com True ou False, indicando se a categoria se aplica ou não. 
# Variáveis com valores booleanos são chamadas de variáveis dummy, e o Pandas tem uma função interna chamada get_dummies(), o qual converte os valores de uma única coluna em múltiplicas colunas de zeros e uns, indicando a presença de uma variável dummy. 

df = pd.DataFrame({
    'cor': ['vermelho', 'azul', 'verde']
})

df_dummies = pd.get_dummies(df)

print(df_dummies)

   cor_azul  cor_verde  cor_vermelho
0     False      False          True
1      True      False         False
2     False       True         False


### Criação de categorias/Conversão de escala

In [84]:
# Existe uma operação de escala, a conversão de escala de um intervalo ou uma proporção, como uma nota numérica, em uma escala categórica. 

# Para isso, vamos usar a função 'cut()' que separa em intervalos. 

import numpy as np

## PREPARAÇÃO
# Lendo o csv
df = pd.read_csv('datasets/census.csv')

# Reduzindo o df apenas para estados
df = df[df['SUMLEV'] == 50]

# Separando em grupos
df = df.set_index('STNAME').groupby(level = 0)['CENSUS2010POP'].agg(np.average)

df.head()

STNAME
Alabama        71339.343284
Alaska         24490.724138
Arizona       426134.466667
Arkansas       38878.906667
California    642309.586207
Name: CENSUS2010POP, dtype: float64

Vamos aplicar a cut().

A função cut() serve para transformar valores numéricos contínuos em categorias (intervalos).

In [ ]:
# Agora vamos aplicar a função cut, a qual vai gerar compartimentos/intervalos
# pd.cut(x, bins, labels = None)
#
# O 'x' pode ser: df['coluna'], uma Series, uma lista ou array
# O primeiro argumento é uma algo de 1D. 
#
# 'bins' define como os intervalos serão criados
# bins = 3 --> teremos 3 intervalos criados automaticamente
# bins=[0, 18, 35, 100] --> intervalos definidos por você: O intervalo é (0, 18], (18, 35]...
# 
# 'labels' é o nome das categorias
# OBS: Número de labels = número de bins

pd.cut(df, 10).head()
# OBS: df aqui é uma série pelo que foi feito anteriormente

STNAME
Alabama         (11706.087, 75333.413]
Alaska          (11706.087, 75333.413]
Arizona       (390320.176, 453317.529]
Arkansas        (11706.087, 75333.413]
California    (579312.234, 642309.586]
Name: CENSUS2010POP, dtype: category
Categories (10, interval[float64, right]): [(11706.087, 75333.413] < (75333.413, 138330.766] < (138330.766, 201328.118] < (201328.118, 264325.471] ... (390320.176, 453317.529] < (453317.529, 516314.881] < (516314.881, 579312.234] < (579312.234, 642309.586]]

In [89]:
# Também podemos fazer: 
pd.cut(df, 3, labels = ['Ruim', 'Médio', 'Bom']).head()


STNAME
Alabama        Ruim
Alaska         Ruim
Arizona       Médio
Arkansas       Ruim
California      Bom
Name: CENSUS2010POP, dtype: category
Categories (3, str): ['Ruim' < 'Médio' < 'Bom']